# CIFAR-10 structured slicing notebook

This notebook contains only the structured slicing experiments used here:

- DSWD
- HDSWD
- HSW_DSW
- HSW_HDSWD

No FEAT / CURV branches are used in this cleaned version.


In [1]:


try:
    import torch_fidelity
except ImportError:
    !pip install -q torch-fidelity
    import torch_fidelity


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 5.7 MB/s eta 0:00:00


In [2]:
# @title
# Optional: install missing packages in Colab
import sys, subprocess, pkgutil

def ensure(pkg, pip_name=None):
    pip_name = pip_name or pkg
    if pkgutil.find_loader(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure("torchmetrics")
ensure("pandas")


/tmp/ipykernel_1981/3939325059.py:7: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  if pkgutil.find_loader(pkg) is None:


In [3]:
# @title
# Mount Google Drive if running in Colab
import os

IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)


Mounted at /content/drive


In [4]:
# @title
import os, json, csv, math, time, random, warnings, uuid
from dataclasses import dataclass, fields
from pathlib import Path
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils as vutils

from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

import re
import shutil
import subprocess
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda


In [5]:
# @title

@dataclass
class TrainConfig:
    experiment_name: str = "cifar10_dsw_v2"
    run_tag: str = "bs512"

    output_root: str = "/content/runs_local"
    drive_output_root: str = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
    data_root: str = "/content/data"

    seeds: Tuple[int, ...] = (42,)
    modes: Tuple[str, ...] = (
        "DSWD",
        "HDSWD",
        "HSW_DSW",
        "HSW_HDSWD",
    )

    image_size: int = 64
    num_channels: int = 3
    latent_size: int = 100
    hidden_channels: int = 64

    batch_size: int = 512
    epochs: int = 100
    lr_g: float = 5e-4
    lr_d: float = 5e-4
    lr_t: float = 1e-4
    beta1: float = 0.5
    beta2: float = 0.999

    num_workers: int = 2
    pin_memory: bool = True

    num_projections: int = 256
    p: int = 2
    selector_steps: int = 1
    selector_hidden: int = 512

    use_diversity_reg: bool = True
    diversity_weight: float = 10.0

    sample_every: int = 5
    eval_every: int = 10
    final_eval_samples: int = 10000
    mid_eval_samples: int = 2000
    kid_subsets: int = 10
    kid_subset_size: int = 100

    resume_if_possible: bool = True
    autoskip_complete: bool = True

    drive_backup_every: int = 10
    drive_backup_on_eval: bool = True
    final_full_sync_to_drive: bool = True
    sync_history_on_backup: bool = True
    sync_eval_history_on_backup: bool = True
    sync_samples_on_backup: bool = False
    sync_eval_ckpts_on_backup: bool = False

    eval_backend: str = "torch_fidelity"
    fidelity_cache_root: str = "/content/fidelity_cache_local"
    fidelity_batch_size: int = 128

    enable_deterministic_algorithms: bool = True

    # hierarchical projector knobs
    h_num_base: int = 128
    h_num_bottleneck: int = 32
    h_base_rank: int = 512
    h_bottleneck_diversity_weight: float = 5.0
    h_mix_act: str = "linear"

    # hierarchical slicer knobs
    hsw_slicer_dims: Tuple[int, ...] = (512, 128)
    hsw_slicer_act: str = "linear"
    hsw_proj_rank: int = 0

    current_epoch: int = 0

cfg = TrainConfig()
print(cfg)


TrainConfig(experiment_name='cifar10_dsw_v2', run_tag='bs512', output_root='/content/runs_local', drive_output_root='/content/drive/MyDrive/cifar10_dsw_runs', data_root='/content/data', seeds=(42,), modes=('DSWD', 'HDSWD', 'HSW_DSW', 'HSW_HDSWD'), image_size=64, num_channels=3, latent_size=100, hidden_channels=64, batch_size=512, epochs=100, lr_g=0.0005, lr_d=0.0005, lr_t=0.0001, beta1=0.5, beta2=0.999, num_workers=2, pin_memory=True, num_projections=256, p=2, selector_steps=1, selector_hidden=512, use_diversity_reg=True, diversity_weight=10.0, sample_every=5, eval_every=10, final_eval_samples=10000, mid_eval_samples=2000, kid_subsets=10, kid_subset_size=100, resume_if_possible=True, autoskip_complete=True, drive_backup_every=10, drive_backup_on_eval=True, final_full_sync_to_drive=True, sync_history_on_backup=True, sync_eval_history_on_backup=True, sync_samples_on_backup=False, sync_eval_ckpts_on_backup=False, eval_backend='torch_fidelity', fidelity_cache_root='/content/fidelity_cache_

In [6]:
# @title
def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    if getattr(cfg, "enable_deterministic_algorithms", True):
        torch.use_deterministic_algorithms(True)

def experiment_root(cfg: TrainConfig) -> Path:
    return Path(cfg.output_root) / cfg.experiment_name / cfg.run_tag

def drive_experiment_root(cfg: TrainConfig) -> Path:
    return Path(cfg.drive_output_root) / cfg.experiment_name / cfg.run_tag

def run_dir(cfg: TrainConfig, mode: str, seed: int) -> Path:
    return experiment_root(cfg) / mode / f"seed_{seed}"

def drive_run_dir(cfg: TrainConfig, mode: str, seed: int) -> Path:
    return drive_experiment_root(cfg) / mode / f"seed_{seed}"

def ensure_run_dirs(base: Path):
    for sub in ["checkpoints", "logs", "samples", "eval", "plots"]:
        (base / sub).mkdir(parents=True, exist_ok=True)

def complete_marker(base: Path) -> Path:
    return base / "COMPLETE_LOCAL"

def synced_complete_marker(base: Path) -> Path:
    return base / "COMPLETE_SYNCED"

def history_csv_path(base: Path) -> Path:
    return base / "logs" / "history.csv"

def eval_csv_path(base: Path) -> Path:
    return base / "logs" / "eval_history.csv"

def drive_last_ckpt_path(cfg: TrainConfig, mode: str, seed: int) -> Path:
    return drive_run_dir(cfg, mode, seed) / "checkpoints" / "last.pt"

def drive_best_ckpt_path(cfg: TrainConfig, mode: str, seed: int) -> Path:
    return drive_run_dir(cfg, mode, seed) / "checkpoints" / "best_fid.pt"

def drive_complete_marker(cfg: TrainConfig, mode: str, seed: int) -> Path:
    return drive_run_dir(cfg, mode, seed) / "COMPLETE"

def config_to_dict(cfg: TrainConfig) -> Dict:
    return dict(vars(cfg))

def append_row(csv_path: Path, row: Dict):
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    exists = csv_path.exists()
    with open(csv_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not exists:
            writer.writeheader()
        writer.writerow(row)

def atomic_write_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp_{uuid.uuid4().hex}")
    with open(tmp, "w", encoding="utf-8") as f:
        f.write(text)
    os.replace(tmp, path)

def save_json(path: Path, obj: Dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp_{uuid.uuid4().hex}")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def atomic_torch_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".tmp_{uuid.uuid4().hex}")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def safe_copy_file(src: Path, dst: Path, verbose: bool = False) -> bool:
    try:
        if not src.exists():
            return False
        dst.parent.mkdir(parents=True, exist_ok=True)
        tmp = dst.with_name(dst.name + f".tmp_{uuid.uuid4().hex}")
        shutil.copy2(src, tmp)
        os.replace(tmp, dst)
        if verbose:
            print(f"[COPIED] {src} -> {dst}")
        return True
    except Exception as e:
        warnings.warn(f"Failed to copy {src} -> {dst}: {e}")
        return False

def sync_resume_artifacts_to_drive(cfg: TrainConfig, mode: str, seed: int, verbose: bool = True) -> bool:
    local_base = run_dir(cfg, mode, seed)
    drive_base = drive_run_dir(cfg, mode, seed)
    drive_base.mkdir(parents=True, exist_ok=True)

    relative_files = [
        Path("checkpoints/last.pt"),
        Path("checkpoints/best_fid.pt"),
        Path("eval/final_metrics.json"),
    ]
    if getattr(cfg, "sync_history_on_backup", True):
        relative_files.append(Path("logs/history.csv"))
    if getattr(cfg, "sync_eval_history_on_backup", True):
        relative_files.append(Path("logs/eval_history.csv"))
    if getattr(cfg, "sync_samples_on_backup", False):
        relative_files.extend([Path("samples/best_grid.png"), Path("samples/last_grid.png")])

    ok = True
    copied = 0
    for rel in relative_files:
        src = local_base / rel
        dst = drive_base / rel
        if src.exists():
            copied += int(safe_copy_file(src, dst, verbose=False))

    if getattr(cfg, "sync_eval_ckpts_on_backup", False):
        eval_ckpt_src = local_base / "checkpoints" / "eval_ckpts"
        eval_ckpt_dst = drive_base / "checkpoints" / "eval_ckpts"
        if eval_ckpt_src.exists():
            for src in sorted(eval_ckpt_src.glob("*.pt")):
                ok = safe_copy_file(src, eval_ckpt_dst / src.name, verbose=False) and ok
                copied += int(ok)

    if verbose:
        print(f"[SYNC-RESUME] {mode} seed={seed} copied sparse artifacts to {drive_base} (count={copied})")
    return ok

def sync_mode_to_drive(cfg: TrainConfig, mode: str, seed: int, verbose: bool = True) -> bool:
    local_base = run_dir(cfg, mode, seed)
    drive_base = drive_run_dir(cfg, mode, seed)
    drive_base.mkdir(parents=True, exist_ok=True)

    ok = True
    copied = 0
    for src in sorted(local_base.rglob("*")):
        if src.is_dir():
            continue
        rel = src.relative_to(local_base)
        dst = drive_base / rel
        ok = safe_copy_file(src, dst, verbose=False) and ok
        copied += int(src.exists())

    if ok:
        atomic_write_text(drive_complete_marker(cfg, mode, seed), "done")
    if verbose:
        status = "OK" if ok else "PARTIAL"
        print(f"[SYNC-FULL:{status}] {local_base} -> {drive_base} (files_seen={copied})")
    return ok

def hydrate_local_run_from_drive(cfg: TrainConfig, mode: str, seed: int):
    local_base = run_dir(cfg, mode, seed)
    drive_base = drive_run_dir(cfg, mode, seed)
    if local_base.exists():
        return
    if drive_base.exists():
        local_base.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(drive_base, local_base, dirs_exist_ok=True)
        print(f"[HYDRATED] {drive_base} -> {local_base}")

def pick_existing_checkpoints(*paths: Path) -> List[Path]:
    existing = [p for p in paths if p.exists()]
    return sorted(existing, key=lambda p: p.stat().st_mtime, reverse=True)

def maybe_sync_completed_local_run(cfg: TrainConfig, mode: str, seed: int) -> bool:
    base = run_dir(cfg, mode, seed)
    if not complete_marker(base).exists():
        return False
    if synced_complete_marker(base).exists() or drive_complete_marker(cfg, mode, seed).exists():
        return True
    if not getattr(cfg, "final_full_sync_to_drive", True):
        return True
    print(f"[INFO] Found locally complete but unsynced run for {mode} seed={seed}; trying final sync only.")
    ok = sync_mode_to_drive(cfg, mode, seed, verbose=True)
    if ok:
        atomic_write_text(synced_complete_marker(base), "done")
    return ok

print("Local experiment root:", experiment_root(cfg))
print("Drive experiment root:", drive_experiment_root(cfg))


Local experiment root: /content/runs_local/cifar10_dsw_v2/bs512
Drive experiment root: /content/drive/MyDrive/cifar10_dsw_runs/cifar10_dsw_v2/bs512


In [7]:
# @title
# Data
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

def build_loaders(cfg: TrainConfig, seed: int):
    g = torch.Generator()
    g.manual_seed(seed)

    tfm = transforms.Compose([
        transforms.Resize(cfg.image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ])

    train_ds = datasets.CIFAR10(cfg.data_root, train=True, download=True, transform=tfm)
    test_ds = datasets.CIFAR10(cfg.data_root, train=False, download=True, transform=tfm)

    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=True,
        worker_init_fn=seed_worker,
        generator=g,
    )
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=g,
    )
    return train_loader, test_loader

train_loader, test_loader = build_loaders(cfg, cfg.seeds[0])
len(train_loader), len(test_loader)


100%|██████████| 170M/170M [00:18<00:00, 9.03MB/s]


(97, 20)

In [8]:
# @title
# Models (adapted from the DSW/DCGAN code)

class Generator(nn.Module):
    def __init__(self, latent_size=128, num_channels=3, hidden_channels=64):
        super().__init__()
        self.latent_size = latent_size
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_size, hidden_channels * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(hidden_channels * 8),
            nn.ReLU(True),

            nn.ConvTranspose2d(hidden_channels * 8, hidden_channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(hidden_channels * 4, hidden_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(hidden_channels * 2, hidden_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(True),

            nn.ConvTranspose2d(hidden_channels, num_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.main(z.view(z.shape[0], self.latent_size, 1, 1))

class Discriminator(nn.Module):
    def __init__(self, latent_size=128, num_channels=3, hidden_channels=64):
        super().__init__()
        self.main1 = nn.Sequential(
            nn.Conv2d(num_channels, hidden_channels, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(hidden_channels, hidden_channels * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(hidden_channels * 2, hidden_channels * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(hidden_channels * 4, hidden_channels * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(hidden_channels * 8),
            nn.Tanh(),
        )
        self.main2 = nn.Sequential(
            nn.Conv2d(hidden_channels * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x):
        feat = self.main1(x)
        y = self.main2(feat).view(x.shape[0], -1)
        return y, feat.view(x.shape[0], -1)

def weights_init(m):
    classname = m.__class__.__name__
    if classname.find("Conv") != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find("BatchNorm") != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

G = Generator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
D = Discriminator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
G.apply(weights_init); D.apply(weights_init)
z = torch.randn(4, cfg.latent_size, device=device)
x = G(z)
y, feat = D(x)
print(x.shape, y.shape, feat.shape)


torch.Size([4, 3, 64, 64]) torch.Size([4, 1]) torch.Size([4, 8192])


In [9]:
# @title
def get_rng_state():
    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_random_state": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["torch_cuda_random_state_all"] = torch.cuda.get_rng_state_all()
    return state


def set_rng_state(state):
    if state is None:
        return

    if "python_random_state" in state:
        random.setstate(state["python_random_state"])

    if "numpy_random_state" in state:
        np.random.set_state(state["numpy_random_state"])

    if "torch_random_state" in state:
        torch.set_rng_state(state["torch_random_state"])

    if torch.cuda.is_available() and "torch_cuda_random_state_all" in state:
        torch.cuda.set_rng_state_all(state["torch_cuda_random_state_all"])


In [10]:
# @title

# Projection nets

class TransformNet(nn.Module):
    def __init__(self, proj_dim: int):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(proj_dim, proj_dim))

    def forward(self, theta, summary=None):
        out = self.net(theta)
        return F.normalize(out, dim=1)


class ConditionedTransformNet(nn.Module):
    def __init__(self, proj_dim: int, summary_dim: int, hidden: int = 512):
        super().__init__()
        self.theta_proj = nn.Linear(proj_dim, hidden)
        self.summary_proj = nn.Linear(summary_dim, hidden)
        self.out = nn.Linear(hidden, proj_dim)

    def forward(self, theta, summary):
        if summary.ndim == 1:
            summary = summary.unsqueeze(0)
        s = self.summary_proj(summary).expand(theta.shape[0], -1)
        h = torch.tanh(self.theta_proj(theta) + s)
        out = self.out(h)
        return F.normalize(out, dim=1)


In [11]:
# @title

# SW / DSW utilities

def sample_random_projections(num_projections: int, dim: int, device: torch.device):
    theta = torch.randn(num_projections, dim, device=device)
    return F.normalize(theta, dim=1)

def sliced_wasserstein_distance(x, y, theta, p=2):
    proj_x = x @ theta.t()
    proj_y = y @ theta.t()
    proj_x = torch.sort(proj_x, dim=0)[0]
    proj_y = torch.sort(proj_y, dim=0)[0]
    diff = (proj_x - proj_y).abs().pow(p)
    return diff.mean().pow(1.0 / p)

def selector_forward(mode: str, selector, theta0, real_feat, fake_feat, real_images=None, fake_images=None, cfg=None):
    return selector(theta0)


In [12]:
# @title

# Train steps

bce = nn.BCELoss()

def set_requires_grad(module: nn.Module, flag: bool):
    if module is None:
        return
    for p in module.parameters():
        p.requires_grad_(flag)

def train_discriminator_step(discriminator, optimizer_d, real_images, fake_images):
    real_label = torch.ones((real_images.shape[0], 1), device=real_images.device)
    fake_label = torch.zeros((real_images.shape[0], 1), device=real_images.device)

    optimizer_d.zero_grad(set_to_none=True)
    y_real, _ = discriminator(real_images.detach())
    loss_real = bce(y_real, real_label)

    y_fake, _ = discriminator(fake_images.detach())
    loss_fake = bce(y_fake, fake_label)

    loss_d = loss_real + loss_fake
    loss_d.backward()
    optimizer_d.step()

    return {
        "d_total": float(loss_d.item()),
        "d_real": float(loss_real.item()),
        "d_fake": float(loss_fake.item()),
    }

def optimize_selector(
    selector,
    opt_t,
    real_feat,
    fake_feat,
    num_projections,
    mode,
    cfg,
    real_images=None,
    fake_images=None,
    p=2,
):
    last = {
        "selector_obj": 0.0,
        "selector_entropy": 0.0,
        "selector_diversity": 0.0,
        "selector_mean_abs_cos": 0.0,
    }

    if selector is None or opt_t is None:
        return last

    for _ in range(cfg.selector_steps):
        theta0 = sample_random_projections(num_projections, real_feat.shape[1], real_feat.device)

        theta = selector_forward(
            mode,
            selector,
            theta0,
            real_feat,
            fake_feat,
            real_images=real_images,
            fake_images=fake_images,
            cfg=cfg,
        )

        sw = sliced_wasserstein_distance(real_feat.detach(), fake_feat.detach(), theta, p=p)

        div_val = torch.tensor(0.0, device=real_feat.device)
        if cfg.use_diversity_reg:
            div_val = direction_diversity_penalty(theta)
            selector_loss = -sw + cfg.diversity_weight * div_val
        else:
            selector_loss = -sw

        mac_val = mean_abs_offdiag_cos(theta)

        opt_t.zero_grad(set_to_none=True)
        selector_loss.backward()
        opt_t.step()

        last = {
            "selector_obj": float(sw.item()),
            "selector_entropy": 0.0,
            "selector_diversity": float(div_val.item()),
            "selector_mean_abs_cos": float(mac_val.item()),
        }

    return last

def generator_step(mode, generator, discriminator, selector, opt_g, real_images, cfg, opt_t=None):
    z = torch.randn(real_images.shape[0], cfg.latent_size, device=real_images.device)
    fake_images = generator(z)

    with torch.no_grad():
        _, real_feat = discriminator(real_images)

    set_requires_grad(discriminator, False)
    try:
        _, fake_feat = discriminator(fake_images)

        selector_stats = optimize_selector(
            selector,
            opt_t,
            real_feat,
            fake_feat,
            cfg.num_projections,
            mode=mode,
            cfg=cfg,
            real_images=real_images,
            fake_images=fake_images,
            p=cfg.p,
        )

        with torch.no_grad():
            theta0 = sample_random_projections(cfg.num_projections, real_feat.shape[1], real_feat.device)
            theta = selector_forward(
                mode,
                selector,
                theta0,
                real_feat,
                fake_feat,
                real_images=real_images,
                fake_images=fake_images,
                cfg=cfg,
            )

        sw = sliced_wasserstein_distance(real_feat, fake_feat, theta, p=cfg.p)
        g_total = sw

        opt_g.zero_grad(set_to_none=True)
        g_total.backward()
        opt_g.step()
    finally:
        set_requires_grad(discriminator, True)

    return fake_images.detach(), {
        "g_total": float(g_total.item()),
        "g_sw": float(sw.item()),
        "selector_obj": float(selector_stats["selector_obj"]),
        "selector_entropy": float(selector_stats["selector_entropy"]),
        "selector_diversity": float(selector_stats["selector_diversity"]),
        "selector_mean_abs_cos": float(selector_stats["selector_mean_abs_cos"]),
    }


In [13]:
# @title

# Checkpoint / resume
from pathlib import Path

def get_selector_arch(mode: str, cfg: TrainConfig, selector=None) -> str:
    cls_name = selector.__class__.__name__ if selector is not None else "None"
    if mode == "DSWD":
        return "DSWD::TransformNet::full"
    return f"{mode}::{cls_name}"

def make_models(cfg: TrainConfig, mode: str):
    G = Generator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
    D = Discriminator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    feat_dim = cfg.hidden_channels * 8 * 4 * 4

    if mode == "DSWD":
        selector = TransformNet(feat_dim).to(device)
    else:
        raise ValueError(mode)

    opt_t = torch.optim.Adam(selector.parameters(), lr=cfg.lr_t, betas=(cfg.beta1, cfg.beta2))
    opt_g = torch.optim.Adam(G.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
    opt_d = torch.optim.Adam(D.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))
    return G, D, selector, opt_g, opt_d, opt_t

def save_eval_checkpoint(
    run_dir,
    epoch,
    mode,
    seed,
    generator,
    discriminator,
    selector,
    opt_g=None,
    opt_d=None,
    opt_t=None,
    metrics=None,
    cfg=None,
):
    ckpt_dir = Path(run_dir) / "checkpoints" / "eval_ckpts"
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    name = f"epoch_{epoch:04d}"
    if metrics is not None and "fid" in metrics:
        try:
            name += f"_fid_{float(metrics['fid']):.3f}"
        except Exception:
            pass
    path = ckpt_dir / f"{name}.pt"

    payload = {
        "run_schema_version": 2,
        "epoch": epoch,
        "mode": mode,
        "seed": seed,
        "selector_arch": get_selector_arch(mode, cfg, selector),
        "generator": generator.state_dict(),
        "discriminator": discriminator.state_dict(),
        "selector": selector.state_dict() if selector is not None else None,
        "metrics": metrics,
        "cfg": config_to_dict(cfg) if cfg is not None else None,
        "rng_state": get_rng_state(),
    }

    if opt_g is not None:
        payload["opt_g"] = opt_g.state_dict()
    if opt_d is not None:
        payload["opt_d"] = opt_d.state_dict()
    if opt_t is not None:
        payload["opt_t"] = opt_t.state_dict()

    atomic_torch_save(payload, path)
    return path

def save_checkpoint(path: Path, epoch: int, G, D, selector, opt_g, opt_d, opt_t, fixed_noise, best_fid, cfg, mode: str, seed: int):
    payload = {
        "run_schema_version": 2,
        "epoch": epoch,
        "mode": mode,
        "seed": seed,
        "selector_arch": get_selector_arch(mode, cfg, selector),
        "G": G.state_dict(),
        "D": D.state_dict(),
        "opt_g": opt_g.state_dict(),
        "opt_d": opt_d.state_dict(),
        "fixed_noise": fixed_noise.detach().cpu(),
        "best_fid": best_fid,
        "cfg": config_to_dict(cfg),
        "rng_state": get_rng_state(),
    }
    if selector is not None:
        payload["selector"] = selector.state_dict()
    if opt_t is not None:
        payload["opt_t"] = opt_t.state_dict()
    atomic_torch_save(payload, path)

def _load_checkpoint_into_objects(ckpt, G, D, selector, opt_g, opt_d, opt_t, expected_mode: str, expected_seed: int, expected_arch: str):
    ckpt_mode = ckpt.get("mode")
    if ckpt_mode is not None and ckpt_mode != expected_mode:
        raise RuntimeError(f"Checkpoint mode mismatch: expected {expected_mode}, got {ckpt_mode}")

    ckpt_seed = ckpt.get("seed")
    if ckpt_seed is not None and int(ckpt_seed) != int(expected_seed):
        raise RuntimeError(f"Checkpoint seed mismatch: expected {expected_seed}, got {ckpt_seed}")

    ckpt_arch = ckpt.get("selector_arch")
    if ckpt_arch is not None and ckpt_arch != expected_arch:
        raise RuntimeError(f"Selector architecture mismatch: expected {expected_arch}, got {ckpt_arch}")

    G.load_state_dict(ckpt["G"])
    D.load_state_dict(ckpt["D"])
    opt_g.load_state_dict(ckpt["opt_g"])
    opt_d.load_state_dict(ckpt["opt_d"])
    if selector is not None and "selector" in ckpt and ckpt["selector"] is not None:
        selector.load_state_dict(ckpt["selector"])
    if opt_t is not None and "opt_t" in ckpt:
        opt_t.load_state_dict(ckpt["opt_t"])

def maybe_resume(base: Path, cfg: TrainConfig, mode: str, seed: int):
    ensure_run_dirs(base)
    G, D, selector, opt_g, opt_d, opt_t = make_models(cfg, mode)
    fixed_noise = torch.randn(64, cfg.latent_size, device=device)
    best_fid = float("inf")
    start_epoch = 1

    expected_arch = get_selector_arch(mode, cfg, selector)
    local_last = base / "checkpoints" / "last.pt"
    drive_last = drive_last_ckpt_path(cfg, mode, seed)

    candidates = []
    for p in [local_last, drive_last]:
        if p.exists():
            candidates.append((p.stat().st_mtime, p))
    candidates.sort(reverse=True)

    loaded = False
    for _, ckpt_path in candidates:
        try:
            ckpt = torch.load(ckpt_path, map_location=device)
            _load_checkpoint_into_objects(ckpt, G, D, selector, opt_g, opt_d, opt_t, mode, seed, expected_arch)
            fixed_noise = ckpt.get("fixed_noise", fixed_noise).to(device)
            best_fid = float(ckpt.get("best_fid", best_fid))
            start_epoch = int(ckpt["epoch"]) + 1
            rng_state = ckpt.get("rng_state")
            if rng_state is not None:
                set_rng_state(rng_state)
            print(f"[INFO] Resumed {mode} seed={seed} from {ckpt_path} @ epoch {start_epoch-1}")
            loaded = True
            break
        except Exception as e:
            warnings.warn(f"Failed to resume from {ckpt_path}: {e}")

    if not loaded:
        print(f"[INFO] No valid checkpoint found for {mode} seed={seed}; starting fresh.")

    return G, D, selector, opt_g, opt_d, opt_t, fixed_noise, start_epoch, best_fid


In [14]:
# @title

# Evaluation and visualization

def save_uint8_tensor_dir(images_uint8: torch.Tensor, out_dir: Path, overwrite: bool = True):
    if overwrite and out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, img in enumerate(images_uint8):
        arr = img.permute(1, 2, 0).cpu().numpy()   # CHW -> HWC
        Image.fromarray(arr).save(out_dir / f"{i:06d}.png")


def get_fidelity_cache_root(cfg: TrainConfig) -> Path:
    root = Path(cfg.fidelity_cache_root)
    root.mkdir(parents=True, exist_ok=True)
    return root

@torch.no_grad()
def save_sample_grid(path: Path, generator, fixed_noise):
    generator.eval()
    samples = generator(fixed_noise.to(device)).cpu()
    grid = vutils.make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    path.parent.mkdir(parents=True, exist_ok=True)
    vutils.save_image(grid, path)

@torch.no_grad()
def collect_generated_images(generator, total_samples: int, cfg: TrainConfig):
    generator.eval()
    batches = []
    remaining = total_samples
    while remaining > 0:
        bs = min(cfg.batch_size, remaining)
        z = torch.randn(bs, cfg.latent_size, device=device)
        fake = generator(z)
        fake_uint8 = ((fake.clamp(-1, 1) + 1) * 127.5).to(torch.uint8).cpu()
        batches.append(fake_uint8)
        remaining -= bs
    return torch.cat(batches, dim=0)

@torch.no_grad()
def collect_real_images(loader, total_samples: int):
    batches = []
    remaining = total_samples
    for x, _ in loader:
        take = min(x.shape[0], remaining)
        x = x[:take]
        x_uint8 = ((x.clamp(-1, 1) + 1) * 127.5).to(torch.uint8)
        batches.append(x_uint8.cpu())
        remaining -= take
        if remaining <= 0:
            break
    return torch.cat(batches, dim=0)

@torch.no_grad()
def evaluate_fid_kid(generator, real_loader, cfg: TrainConfig, total_samples: int, mode: str, seed: int):
    rng_state = get_rng_state()
    cache_root = get_fidelity_cache_root(cfg)
    fake_dir = None
    try:
        real = collect_real_images(real_loader, total_samples)
        fake = collect_generated_images(generator, total_samples, cfg)

        real_dir = cache_root / cfg.run_tag / "real_cache" / f"img_{cfg.image_size}" / f"n_{total_samples}"
        if not real_dir.exists() or len(list(real_dir.glob("*.png"))) != total_samples:
            save_uint8_tensor_dir(real, real_dir, overwrite=True)

        fake_dir = cache_root / cfg.run_tag / "_tmp_fake" / mode / f"seed_{seed}" / f"eval_{total_samples}_{uuid.uuid4().hex}"
        save_uint8_tensor_dir(fake, fake_dir, overwrite=True)

        metrics_dict = torch_fidelity.calculate_metrics(
            input1=str(fake_dir),
            input2=str(real_dir),
            cuda=(device.type == "cuda"),
            fid=True,
            kid=True,
            isc=False,
            prc=False,
            verbose=False,
            cache_root=str(cache_root),
            batch_size=cfg.fidelity_batch_size,
            samples_find_deep=False,
        )

        return {
            "fid": float(metrics_dict["frechet_inception_distance"]),
            "kid_mean": float(metrics_dict["kernel_inception_distance_mean"]),
            "kid_std": float(metrics_dict["kernel_inception_distance_std"]),
        }

    finally:
        if fake_dir is not None and fake_dir.exists():
            shutil.rmtree(fake_dir, ignore_errors=True)
        set_rng_state(rng_state)

def make_run_plots(base: Path):
    hist_path = history_csv_path(base)
    eval_path = eval_csv_path(base)
    if not hist_path.exists():
        return
    hist = pd.read_csv(hist_path)
    eval_df = pd.read_csv(eval_path) if eval_path.exists() else pd.DataFrame()

    plot_dir = base / "plots"
    plot_dir.mkdir(parents=True, exist_ok=True)

    plt.figure(figsize=(10, 5))
    for col in ["g_total", "g_sw", "d_total"]:
        if col in hist.columns:
            plt.plot(hist["epoch"], hist[col], label=col)
    plt.legend()
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title("Training losses")
    plt.tight_layout()
    plt.savefig(plot_dir / "losses.png", dpi=160)
    plt.close()

    if "selector_diversity" in hist.columns:
        plt.figure(figsize=(8, 4))
        plt.plot(hist["epoch"], hist["selector_diversity"], label="selector_diversity")
        plt.xlabel("epoch")
        plt.ylabel("penalty")
        plt.title("Selector diversity penalty")
        plt.tight_layout()
        plt.savefig(plot_dir / "selector_diversity.png", dpi=160)
        plt.close()

    if "selector_mean_abs_cos" in hist.columns:
        plt.figure(figsize=(8, 4))
        plt.plot(hist["epoch"], hist["selector_mean_abs_cos"], label="selector_mean_abs_cos")
        plt.xlabel("epoch")
        plt.ylabel("mean abs cos")
        plt.title("Mean absolute off-diagonal cosine")
        plt.tight_layout()
        plt.savefig(plot_dir / "selector_mean_abs_cos.png", dpi=160)
        plt.close()

    if len(eval_df) > 0:
        plt.figure(figsize=(10, 4))
        plt.subplot(1, 2, 1)
        plt.plot(eval_df["epoch"], eval_df["fid"], marker="o")
        plt.title("FID")
        plt.xlabel("epoch")
        plt.subplot(1, 2, 2)
        plt.plot(eval_df["epoch"], eval_df["kid_mean"], marker="o")
        plt.title("KID")
        plt.xlabel("epoch")
        plt.tight_layout()
        plt.savefig(plot_dir / "fid_kid.png", dpi=160)
        plt.close()


In [15]:
# @title
# Training loop

def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def train_one_run(cfg: TrainConfig, mode: str, seed: int):
    hydrate_local_run_from_drive(cfg, mode, seed)

    base = run_dir(cfg, mode, seed)
    ensure_run_dirs(base)

    if cfg.autoskip_complete:
        if synced_complete_marker(base).exists() or drive_complete_marker(cfg, mode, seed).exists():
            print(f"[SKIP] {mode} seed={seed} already complete and synced.")
            return
        if maybe_sync_completed_local_run(cfg, mode, seed):
            print(f"[SKIP] {mode} seed={seed} already complete locally; sync handled.")
            return

    seed_everything(seed)
    train_loader, test_loader = build_loaders(cfg, seed)
    G, D, selector, opt_g, opt_d, opt_t, fixed_noise, start_epoch, best_fid = maybe_resume(base, cfg, mode, seed)
    last_eval_metrics = None
    last_eval_epoch = None

    for epoch in range(start_epoch, cfg.epochs + 1):
        cfg.current_epoch = epoch
        G.train()
        D.train()
        meter = {k: [] for k in [
            "g_total",
            "g_sw",
            "d_total",
            "d_real",
            "d_fake",
            "selector_obj",
            "selector_entropy",
            "selector_diversity",
            "selector_mean_abs_cos",
        ]}

        t0 = time.time()
        batch_compute_times = []
        total_samples_this_epoch = 0

        for real_images, _ in train_loader:
            real_images = real_images.to(device, non_blocking=True)
            bs = real_images.shape[0]
            total_samples_this_epoch += bs

            sync_device(device)
            batch_t0 = time.perf_counter()

            z = torch.randn(bs, cfg.latent_size, device=device)
            fake_images = G(z).detach()

            d_stats = train_discriminator_step(D, opt_d, real_images, fake_images)
            fake_images, g_stats = generator_step(mode, G, D, selector, opt_g, real_images, cfg, opt_t=opt_t)

            sync_device(device)
            batch_t1 = time.perf_counter()
            batch_compute_times.append(batch_t1 - batch_t0)

            for k, v in d_stats.items():
                meter[k].append(v)
            for k, v in g_stats.items():
                meter[k].append(v)

        epoch_time_sec = time.time() - t0
        total_compute_time = float(np.sum(batch_compute_times))
        avg_minibatch_compute_sec = float(np.mean(batch_compute_times))
        std_minibatch_compute_sec = float(np.std(batch_compute_times))
        throughput_img_per_sec = float(total_samples_this_epoch / total_compute_time) if total_compute_time > 0 else 0.0

        row = {
            "epoch": epoch,
            "seed": seed,
            "mode": mode,
            "time_sec": round(epoch_time_sec, 3),
            "num_minibatches": len(batch_compute_times),
            "total_compute_sec": total_compute_time,
            "avg_minibatch_compute_sec": avg_minibatch_compute_sec,
            "std_minibatch_compute_sec": std_minibatch_compute_sec,
            "throughput_img_per_sec": throughput_img_per_sec,
            "g_total": float(np.mean(meter["g_total"])),
            "g_sw": float(np.mean(meter["g_sw"])),
            "d_total": float(np.mean(meter["d_total"])),
            "d_real": float(np.mean(meter["d_real"])),
            "d_fake": float(np.mean(meter["d_fake"])),
            "selector_obj": float(np.mean(meter["selector_obj"])),
            "selector_entropy": float(np.mean(meter["selector_entropy"])),
            "selector_diversity": float(np.mean(meter["selector_diversity"])),
            "selector_mean_abs_cos": float(np.mean(meter["selector_mean_abs_cos"])),
            "lr_g": opt_g.param_groups[0]["lr"],
            "lr_d": opt_d.param_groups[0]["lr"],
            "lr_t": opt_t.param_groups[0]["lr"] if opt_t is not None else 0.0,
        }
        append_row(history_csv_path(base), row)

        print(
            f"[{mode}][seed={seed}] epoch {epoch:03d}/{cfg.epochs} | "
            f"G={row['g_total']:.4f} D={row['d_total']:.4f} SW={row['g_sw']:.4f} "
            f"DIV={row['selector_diversity']:.4f} "
            f"MAC={row['selector_mean_abs_cos']:.4f} "
            f"MB_time={row['avg_minibatch_compute_sec']:.4f}s"
        )

        if epoch % cfg.sample_every == 0 or epoch == 1:
            save_sample_grid(base / "samples" / f"epoch_{epoch:04d}.png", G, fixed_noise)

        if epoch % cfg.eval_every == 0 or epoch == cfg.epochs:
            metrics = evaluate_fid_kid(
                G,
                test_loader,
                cfg,
                total_samples=cfg.mid_eval_samples if epoch < cfg.epochs else cfg.final_eval_samples,
                mode=mode,
                seed=seed,
            )
            eval_row = {"epoch": epoch, "seed": seed, "mode": mode, **metrics}
            append_row(eval_csv_path(base), eval_row)
            print("  eval:", metrics)
            last_eval_metrics = metrics
            last_eval_epoch = epoch

            save_eval_checkpoint(
                run_dir=run_dir(cfg, mode, seed),
                epoch=epoch,
                mode=mode,
                seed=seed,
                generator=G,
                discriminator=D,
                selector=selector,
                opt_g=opt_g,
                opt_d=opt_d,
                opt_t=opt_t,
                metrics=metrics,
                cfg=cfg,
            )

            if metrics["fid"] < best_fid:
                best_fid = metrics["fid"]
                save_checkpoint(
                    base / "checkpoints" / "best_fid.pt",
                    epoch,
                    G,
                    D,
                    selector,
                    opt_g,
                    opt_d,
                    opt_t,
                    fixed_noise,
                    best_fid,
                    cfg,
                    mode=mode,
                    seed=seed,
                )
                save_sample_grid(base / "samples" / "best_grid.png", G, fixed_noise)

            if getattr(cfg, "drive_backup_on_eval", True):
                sync_resume_artifacts_to_drive(cfg, mode, seed, verbose=True)

        save_checkpoint(
            base / "checkpoints" / "last.pt",
            epoch,
            G,
            D,
            selector,
            opt_g,
            opt_d,
            opt_t,
            fixed_noise,
            best_fid,
            cfg,
            mode=mode,
            seed=seed,
        )

        if cfg.drive_backup_every > 0 and (epoch % cfg.drive_backup_every == 0):
            sync_resume_artifacts_to_drive(cfg, mode, seed, verbose=True)

    if last_eval_metrics is None or last_eval_epoch != cfg.epochs:
        final_metrics = evaluate_fid_kid(
            G,
            test_loader,
            cfg,
            total_samples=cfg.final_eval_samples,
            mode=mode,
            seed=seed,
        )
    else:
        final_metrics = last_eval_metrics

    save_json(base / "eval" / "final_metrics.json", final_metrics)
    save_sample_grid(base / "samples" / "last_grid.png", G, fixed_noise)
    make_run_plots(base)
    atomic_write_text(complete_marker(base), "done")

    sync_resume_artifacts_to_drive(cfg, mode, seed, verbose=True)
    if getattr(cfg, "final_full_sync_to_drive", True):
        ok = sync_mode_to_drive(cfg, mode, seed, verbose=True)
        if ok:
            atomic_write_text(synced_complete_marker(base), "done")
    else:
        atomic_write_text(synced_complete_marker(base), "done")
    try:
        from google.colab import drive
        drive.flush_and_unmount()
        print("[FLUSH] Drive 已强制同步完成")
    except Exception as e:
        print(f"[WARN] flush_and_unmount 失败: {e}")
    print(f"[DONE] {mode} seed={seed} -> {base}")


def train_all(cfg: TrainConfig):
    for mode in cfg.modes:
        for seed in cfg.seeds:
            train_one_run(cfg, mode, seed)


## Patch: structured modes with two switches

This cleaned notebook uses two structural switches:

- hierarchical slicer encoder: OFF / ON
- hierarchical projector: OFF / ON

This gives four modes:

- `DSWD`      : encoder OFF, projector OFF
- `HDSWD`     : encoder OFF, projector ON
- `HSW_DSW`   : encoder ON,  projector OFF
- `HSW_HDSWD` : encoder ON,  projector ON


In [16]:

# @title
# Unified structured selectors: optional hierarchical slicer + optional hierarchical projector

_old_get_selector_arch = get_selector_arch
_old_make_models = make_models
_old_selector_forward = selector_forward
_old_optimize_selector = optimize_selector
_old_generator_step = generator_step


class LowRankTransformNet(nn.Module):
    def __init__(self, proj_dim: int, rank: int = 512):
        super().__init__()
        self.rank = int(rank)
        self.down = nn.Linear(proj_dim, self.rank, bias=False)
        self.up = nn.Linear(self.rank, proj_dim, bias=False)
        nn.init.normal_(self.down.weight, mean=0.0, std=1.0 / math.sqrt(proj_dim))
        nn.init.zeros_(self.up.weight)

    def forward(self, theta):
        out = theta + self.up(self.down(theta))
        return F.normalize(out, dim=1)


class PersistentHierarchicalSlicer(nn.Module):
    def __init__(self, d: int, Ls, act: str = "linear"):
        super().__init__()
        self.d = int(d)
        self.Ls = list(Ls)
        self.act = str(act)

        modules = []
        for i, out_dim in enumerate(self.Ls):
            in_dim = self.d if i == 0 else self.Ls[i - 1]
            modules.append(nn.Linear(in_dim, out_dim, bias=False))
            if i != len(self.Ls) - 1 and self.act == "tanh":
                modules.append(nn.Tanh())
        self.U_list = nn.Sequential(*modules)
        self.reset()

    @property
    def out_dim(self):
        return int(self.Ls[-1])

    def forward(self, x):
        x = x.view(x.shape[0], -1)
        return self.U_list(x)

    def project_parameters(self):
        with torch.no_grad():
            for U in self.U_list.modules():
                if isinstance(U, nn.Linear):
                    U.weight.copy_(U.weight / U.weight.norm(dim=1, keepdim=True).clamp_min(1e-8))

    def reset(self):
        with torch.no_grad():
            for U in self.U_list.modules():
                if isinstance(U, nn.Linear):
                    nn.init.normal_(U.weight, mean=0.0, std=1.0)
            self.project_parameters()


class HierarchicalProjector(nn.Module):
    def __init__(
        self,
        proj_dim: int,
        num_base: int = 128,
        num_bottleneck: int = 32,
        num_final: int = 256,
        base_rank: int = 512,
        act: str = "linear",
    ):
        super().__init__()
        self.proj_dim = int(proj_dim)
        self.num_base = int(num_base)
        self.num_bottleneck = int(num_bottleneck)
        self.num_final = int(num_final)
        self.base_rank = int(base_rank)
        self.act = str(act)

        if self.base_rank > 0 and self.base_rank < proj_dim:
            self.base_transform = LowRankTransformNet(proj_dim, rank=self.base_rank)
        else:
            self.base_transform = TransformNet(proj_dim)

        self.mix1 = nn.Parameter(torch.randn(self.num_bottleneck, self.num_base) * 0.05)
        self.mix2 = nn.Parameter(torch.randn(self.num_final, self.num_bottleneck) * 0.05)

    def _row_normalize(self, W: torch.Tensor, eps: float = 1e-8):
        return W / W.norm(dim=1, keepdim=True).clamp_min(eps)

    def project_parameters(self):
        with torch.no_grad():
            self.mix1.copy_(self._row_normalize(self.mix1))
            self.mix2.copy_(self._row_normalize(self.mix2))

    def reset(self):
        with torch.no_grad():
            self.mix1.normal_(0.0, 0.05)
            self.mix2.normal_(0.0, 0.05)
            self.project_parameters()

    def forward(self, theta0: torch.Tensor, return_intermediates: bool = False):
        theta_base = self.base_transform(theta0)

        W1 = self._row_normalize(self.mix1)
        bottleneck = W1 @ theta_base
        if self.act == "tanh":
            bottleneck = torch.tanh(bottleneck)
        bottleneck = F.normalize(bottleneck, dim=1)

        W2 = self._row_normalize(self.mix2)
        theta_final = W2 @ bottleneck
        theta_final = F.normalize(theta_final, dim=1)

        if return_intermediates:
            return theta_final, bottleneck, theta_base
        return theta_final


class StructuredSelector(nn.Module):
    def __init__(
        self,
        feat_dim: int,
        use_hier_slicer: bool = False,
        use_hier_projector: bool = False,
        slicer_dims=None,
        slicer_act: str = "linear",
        proj_rank: int = 0,
        num_base: int = 128,
        num_bottleneck: int = 32,
        num_final: int = 256,
        base_rank: int = 512,
        hier_proj_act: str = "linear",
    ):
        super().__init__()
        self.feat_dim = int(feat_dim)
        self.use_hier_slicer = bool(use_hier_slicer)
        self.use_hier_projector = bool(use_hier_projector)

        self.slicer_dims = list(slicer_dims or [512, 128])
        self.slicer_act = str(slicer_act)
        self.proj_rank = int(proj_rank)

        self.num_base = int(num_base)
        self.num_bottleneck = int(num_bottleneck)
        self.num_final = int(num_final)
        self.base_rank = int(base_rank)
        self.hier_proj_act = str(hier_proj_act)

        if self.use_hier_slicer:
            self.slicer = PersistentHierarchicalSlicer(self.feat_dim, self.slicer_dims, act=self.slicer_act)
            rep_dim = self.slicer.out_dim
        else:
            self.slicer = None
            rep_dim = self.feat_dim
        self._rep_dim = int(rep_dim)

        if self.use_hier_projector:
            self.projector = HierarchicalProjector(
                proj_dim=self._rep_dim,
                num_base=self.num_base,
                num_bottleneck=self.num_bottleneck,
                num_final=self.num_final,
                base_rank=self.base_rank,
                act=self.hier_proj_act,
            )
        else:
            if self.proj_rank > 0 and self.proj_rank < self._rep_dim:
                self.projector = LowRankTransformNet(self._rep_dim, rank=self.proj_rank)
            else:
                self.projector = TransformNet(self._rep_dim)

    @property
    def rep_dim(self):
        return self._rep_dim

    def encode(self, feat: torch.Tensor):
        if self.slicer is None:
            return feat.view(feat.shape[0], -1)
        return self.slicer(feat)

    def project_theta(self, theta0: torch.Tensor, return_intermediates: bool = False):
        if self.use_hier_projector:
            return self.projector(theta0, return_intermediates=return_intermediates)
        theta = self.projector(theta0)
        if return_intermediates:
            return theta, None, None
        return theta

    def project_parameters(self):
        if self.slicer is not None and hasattr(self.slicer, "project_parameters"):
            self.slicer.project_parameters()
        if self.use_hier_projector and hasattr(self.projector, "project_parameters"):
            self.projector.project_parameters()


In [17]:

# @title
# Override selector architecture / model construction / selector forward for all structured modes

STRUCTURED_MODES = ("HDSWD", "HSW_DSW", "HSW_HDSWD")

def get_selector_arch(mode: str, cfg: TrainConfig, selector=None) -> str:
    cls_name = selector.__class__.__name__ if selector is not None else "None"
    if cls_name == "StructuredSelector":
        return (
            f"{mode}::StructuredSelector::"
            f"slicer_{getattr(selector, 'use_hier_slicer', False)}::"
            f"projector_{getattr(selector, 'use_hier_projector', False)}::"
            f"slicer_dims_{getattr(selector, 'slicer_dims', None)}::"
            f"slicer_act_{getattr(selector, 'slicer_act', None)}::"
            f"proj_rank_{getattr(selector, 'proj_rank', None)}::"
            f"num_base_{getattr(selector, 'num_base', None)}::"
            f"num_bottleneck_{getattr(selector, 'num_bottleneck', None)}::"
            f"num_final_{getattr(selector, 'num_final', None)}::"
            f"base_rank_{getattr(selector, 'base_rank', None)}::"
            f"hier_proj_act_{getattr(selector, 'hier_proj_act', None)}"
        )
    return _old_get_selector_arch(mode, cfg, selector)

def make_models(cfg: TrainConfig, mode: str):
    if mode not in STRUCTURED_MODES:
        return _old_make_models(cfg, mode)

    G = Generator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
    D = Discriminator(cfg.latent_size, cfg.num_channels, cfg.hidden_channels).to(device)
    G.apply(weights_init)
    D.apply(weights_init)

    feat_dim = cfg.hidden_channels * 8 * 4 * 4

    use_hier_slicer = mode in ("HSW_DSW", "HSW_HDSWD")
    use_hier_projector = mode in ("HDSWD", "HSW_HDSWD")

    selector = StructuredSelector(
        feat_dim=feat_dim,
        use_hier_slicer=use_hier_slicer,
        use_hier_projector=use_hier_projector,
        slicer_dims=list(getattr(cfg, "hsw_slicer_dims", [512, 128])),
        slicer_act=str(getattr(cfg, "hsw_slicer_act", "linear")),
        proj_rank=int(getattr(cfg, "hsw_proj_rank", 0)),
        num_base=int(getattr(cfg, "h_num_base", 128)),
        num_bottleneck=int(getattr(cfg, "h_num_bottleneck", 32)),
        num_final=int(getattr(cfg, "num_projections", 256)),
        base_rank=int(getattr(cfg, "h_base_rank", 512)),
        hier_proj_act=str(getattr(cfg, "h_mix_act", "linear")),
    ).to(device)
    selector.project_parameters()

    opt_t = torch.optim.Adam(selector.parameters(), lr=cfg.lr_t, betas=(cfg.beta1, cfg.beta2))
    opt_g = torch.optim.Adam(G.parameters(), lr=cfg.lr_g, betas=(cfg.beta1, cfg.beta2))
    opt_d = torch.optim.Adam(D.parameters(), lr=cfg.lr_d, betas=(cfg.beta1, cfg.beta2))
    return G, D, selector, opt_g, opt_d, opt_t

def selector_forward(mode: str, selector, theta0, real_feat, fake_feat, real_images=None, fake_images=None, cfg=None):
    if mode in STRUCTURED_MODES and isinstance(selector, StructuredSelector):
        return selector.project_theta(theta0, return_intermediates=False)

    return _old_selector_forward(
        mode,
        selector,
        theta0,
        real_feat,
        fake_feat,
        real_images=real_images,
        fake_images=fake_images,
        cfg=cfg,
    )


In [18]:
# @title
# Override diversity penalty: off-diagonal only

def direction_diversity_penalty(theta: torch.Tensor, eps: float = 1e-8):
    theta = F.normalize(theta, dim=1)
    cos_mat = theta @ theta.t()
    L = theta.shape[0]
    eye = torch.eye(L, device=theta.device, dtype=theta.dtype)
    off = cos_mat - eye
    return off.pow(2).sum() / (L * (L - 1) + eps)

def mean_abs_offdiag_cos(theta: torch.Tensor):
    theta = F.normalize(theta, dim=1)
    gram = theta @ theta.t()
    L = theta.shape[0]
    eye = torch.eye(L, device=theta.device, dtype=theta.dtype)
    off = (gram - eye).abs()
    return off.sum() / (L * (L - 1) + 1e-8)


In [19]:

# @title
# Override selector optimization + generator step for all structured modes

def _structured_representation(selector, feat):
    return selector.encode(feat)

def _structured_theta0(selector, num_projections, device):
    if getattr(selector, "use_hier_projector", False):
        n = int(getattr(selector, "num_base", num_projections))
    else:
        n = int(num_projections)
    return sample_random_projections(n, selector.rep_dim, device)

def optimize_selector(
    selector,
    opt_t,
    real_feat,
    fake_feat,
    num_projections,
    mode,
    cfg,
    real_images=None,
    fake_images=None,
    p=2,
):
    if not (mode in STRUCTURED_MODES and isinstance(selector, StructuredSelector)):
        return _old_optimize_selector(
            selector,
            opt_t,
            real_feat,
            fake_feat,
            num_projections,
            mode,
            cfg,
            real_images=real_images,
            fake_images=fake_images,
            p=p,
        )

    last = {
        "selector_obj": 0.0,
        "selector_entropy": 0.0,
        "selector_diversity": 0.0,
        "selector_mean_abs_cos": 0.0,
    }

    div_w_final = float(getattr(cfg, "diversity_weight", 10.0))
    div_w_bottleneck = float(getattr(cfg, "h_bottleneck_diversity_weight", 5.0))

    for _ in range(cfg.selector_steps):
        Zx = _structured_representation(selector, real_feat.detach())
        Zy = _structured_representation(selector, fake_feat.detach())

        theta0 = _structured_theta0(selector, num_projections, real_feat.device)
        theta_final, bottleneck, theta_base = selector.project_theta(theta0, return_intermediates=True)

        sw = sliced_wasserstein_distance(Zx, Zy, theta_final, p=p)

        div_final = torch.tensor(0.0, device=real_feat.device)
        div_bottleneck = torch.tensor(0.0, device=real_feat.device)

        if cfg.use_diversity_reg:
            div_final = direction_diversity_penalty(theta_final)
            if bottleneck is not None:
                div_bottleneck = direction_diversity_penalty(bottleneck)
            selector_loss = -sw + div_w_final * div_final + div_w_bottleneck * div_bottleneck
        else:
            selector_loss = -sw

        mac_val = mean_abs_offdiag_cos(theta_final)

        opt_t.zero_grad(set_to_none=True)
        selector_loss.backward()
        opt_t.step()

        if hasattr(selector, "project_parameters"):
            selector.project_parameters()

        last = {
            "selector_obj": float(sw.item()),
            "selector_entropy": 0.0,
            "selector_diversity": float((div_final + div_w_bottleneck * div_bottleneck).item()),
            "selector_mean_abs_cos": float(mac_val.item()),
        }

    return last

def generator_step(mode, generator, discriminator, selector, opt_g, real_images, cfg, opt_t=None):
    if not (mode in STRUCTURED_MODES and isinstance(selector, StructuredSelector)):
        return _old_generator_step(mode, generator, discriminator, selector, opt_g, real_images, cfg, opt_t=opt_t)

    z = torch.randn(real_images.shape[0], cfg.latent_size, device=real_images.device)
    fake_images = generator(z)

    with torch.no_grad():
        _, real_feat = discriminator(real_images)

    set_requires_grad(discriminator, False)
    set_requires_grad(selector, False)
    try:
        _, fake_feat = discriminator(fake_images)

        set_requires_grad(selector, True)
        selector_stats = optimize_selector(
            selector,
            opt_t,
            real_feat,
            fake_feat,
            cfg.num_projections,
            mode=mode,
            cfg=cfg,
            real_images=real_images,
            fake_images=fake_images,
            p=cfg.p,
        )
        set_requires_grad(selector, False)

        with torch.no_grad():
            Zx = _structured_representation(selector, real_feat)
            theta0 = _structured_theta0(selector, cfg.num_projections, real_feat.device)
            theta = selector.project_theta(theta0, return_intermediates=False)

        Zy = _structured_representation(selector, fake_feat)
        sw = sliced_wasserstein_distance(Zx, Zy, theta, p=cfg.p)
        g_total = sw

        opt_g.zero_grad(set_to_none=True)
        g_total.backward()
        opt_g.step()
    finally:
        set_requires_grad(discriminator, True)
        set_requires_grad(selector, True)

    return fake_images.detach(), {
        "g_total": float(g_total.item()),
        "g_sw": float(sw.item()),
        "selector_obj": float(selector_stats["selector_obj"]),
        "selector_entropy": float(selector_stats["selector_entropy"]),
        "selector_diversity": float(selector_stats["selector_diversity"]),
        "selector_mean_abs_cos": float(selector_stats["selector_mean_abs_cos"]),
    }


## Run: DSWD / HDSWD / HSW_DSW / HSW_HDSWD

In [21]:
# @title
# Run four structured modes with two switches:
# DSWD (none), HDSWD (projector only), HSW_DSW (slicer only), HSW_HDSWD (both)

cfg = TrainConfig()

cfg.output_root = "/content/runs_local"
cfg.drive_output_root = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
cfg.fidelity_cache_root = "/content/fidelity_cache_local"

cfg.p = 2
cfg.seeds = (42,)
cfg.modes = ("DSWD", "HDSWD", "HSW_DSW", "HSW_HDSWD")

cfg.epochs = 100
cfg.eval_every = 10
cfg.sample_every = 5
cfg.run_tag = "bs512_structured_four_modes_v1"

cfg.latent_size = 100
cfg.lr_g = 5e-4
cfg.lr_d = 5e-4
cfg.lr_t = 1e-4

cfg.num_projections = 256
cfg.selector_steps = 1
cfg.use_diversity_reg = True
cfg.diversity_weight = 10.0

cfg.h_num_base = 128
cfg.h_num_bottleneck = 32
cfg.h_base_rank = 512
cfg.h_bottleneck_diversity_weight = 5.0
cfg.h_mix_act = "linear"

cfg.hsw_slicer_dims = (512, 128)
cfg.hsw_slicer_act = "linear"
cfg.hsw_proj_rank = 0

cfg.drive_backup_every = 10
cfg.drive_backup_on_eval = True
cfg.final_full_sync_to_drive = True
cfg.sync_history_on_backup = True
cfg.sync_eval_history_on_backup = True
cfg.sync_samples_on_backup = False
cfg.sync_eval_ckpts_on_backup = False

cfg.resume_if_possible = True
cfg.autoskip_complete = True

print("Experiment root:", experiment_root(cfg))
train_all(cfg)


Experiment root: /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1
[INFO] No valid checkpoint found for DSWD seed=42; starting fresh.


/tmp/ipykernel_1981/1607171771.py:155: UserWarning: Failed to resume from /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1/DSWD/seed_42/checkpoints/last.pt: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray._reconstruct was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct])` or the `torch.serialization.safe_glob

[DSWD][seed=42] epoch 001/100 | G=1.3431 D=2.1900 SW=1.3431 DIV=0.1471 MAC=0.3769 MB_time=0.0393s
[DSWD][seed=42] epoch 002/100 | G=1.6251 D=0.8590 SW=1.6251 DIV=0.0474 MAC=0.2150 MB_time=0.0388s
[DSWD][seed=42] epoch 003/100 | G=1.5818 D=0.9773 SW=1.5818 DIV=0.0422 MAC=0.2008 MB_time=0.0390s
[DSWD][seed=42] epoch 004/100 | G=1.5149 D=0.9523 SW=1.5149 DIV=0.0364 MAC=0.1856 MB_time=0.0388s
[DSWD][seed=42] epoch 005/100 | G=1.4519 D=0.8869 SW=1.4519 DIV=0.0353 MAC=0.1826 MB_time=0.0389s
[DSWD][seed=42] epoch 006/100 | G=1.4407 D=0.8943 SW=1.4407 DIV=0.0356 MAC=0.1840 MB_time=0.0391s
[DSWD][seed=42] epoch 007/100 | G=1.6287 D=0.7495 SW=1.6287 DIV=0.0392 MAC=0.1937 MB_time=0.0388s
[DSWD][seed=42] epoch 008/100 | G=1.6512 D=0.8373 SW=1.6512 DIV=0.0419 MAC=0.2005 MB_time=0.0388s
[DSWD][seed=42] epoch 009/100 | G=1.5432 D=0.6955 SW=1.5432 DIV=0.0391 MAC=0.1938 MB_time=0.0388s
[DSWD][seed=42] epoch 010/100 | G=1.6801 D=0.7329 SW=1.6801 DIV=0.0408 MAC=0.1980 MB_time=0.0387s


Downloading: "https://github.com/toshas/torch-fidelity/releases/download/v0.2.0/weights-inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/weights-inception-2015-12-05-6726825d.pth


  eval: {'fid': 276.7709660721068, 'kid_mean': 0.24400680303573608, 'kid_std': 0.0019143165088067998}
[SYNC-RESUME] DSWD seed=42 copied sparse artifacts to /content/drive/MyDrive/cifar10_dsw_runs/cifar10_dsw_v2/bs512_structured_four_modes_v1/DSWD/seed_42 (count=4)
[SYNC-RESUME] DSWD seed=42 copied sparse artifacts to /content/drive/MyDrive/cifar10_dsw_runs/cifar10_dsw_v2/bs512_structured_four_modes_v1/DSWD/seed_42 (count=4)
[DSWD][seed=42] epoch 011/100 | G=1.5338 D=0.6536 SW=1.5338 DIV=0.0394 MAC=0.1945 MB_time=0.0394s
[DSWD][seed=42] epoch 012/100 | G=1.1856 D=0.8902 SW=1.1856 DIV=0.0330 MAC=0.1774 MB_time=0.0393s
[DSWD][seed=42] epoch 013/100 | G=0.9908 D=0.7828 SW=0.9908 DIV=0.0260 MAC=0.1565 MB_time=0.0394s
[DSWD][seed=42] epoch 014/100 | G=0.8980 D=0.8568 SW=0.8980 DIV=0.0230 MAC=0.1468 MB_time=0.0392s
[DSWD][seed=42] epoch 015/100 | G=0.8558 D=0.8207 SW=0.8558 DIV=0.0212 MAC=0.1403 MB_time=0.0393s
[DSWD][seed=42] epoch 016/100 | G=0.8138 D=0.8701 SW=0.8138 DIV=0.0201 MAC=0.1364 

## Summary

In [22]:
# @title
# Summarize results (best checkpoint and final epoch)
from pathlib import Path
import pandas as pd

base = experiment_root(cfg)
rows_best = []
rows_final = []

for mode_dir in base.iterdir():
    if not mode_dir.is_dir():
        continue
    for seed_dir in mode_dir.iterdir():
        eval_csv = seed_dir / "logs" / "eval_history.csv"
        if not eval_csv.exists():
            continue
        df = pd.read_csv(eval_csv)
        if len(df) == 0:
            continue

        best = df.loc[df["fid"].idxmin()].copy()
        final = df.sort_values("epoch").iloc[-1].copy()

        best["seed"] = seed_dir.name
        final["seed"] = seed_dir.name
        rows_best.append(best)
        rows_final.append(final)

best_df = pd.DataFrame(rows_best).sort_values("fid").reset_index(drop=True)
final_df = pd.DataFrame(rows_final).sort_values("fid").reset_index(drop=True)

print("=== Best over training ===")
display(best_df)

print("\n=== Final epoch ===")
display(final_df)


=== Best over training ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
1,100,seed_42,HDSWD,58.694691,0.041272,0.001359
2,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299
3,100,seed_42,DSWD,62.531611,0.043902,0.001393



=== Final epoch ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
1,100,seed_42,HDSWD,58.694691,0.041272,0.001359
2,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299
3,100,seed_42,DSWD,62.531611,0.043902,0.001393


In [23]:
# @title
# Run four structured modes with two switches:
# DSWD (none), HDSWD (projector only), HSW_DSW (slicer only), HSW_HDSWD (both)

cfg = TrainConfig()

cfg.output_root = "/content/runs_local"
cfg.drive_output_root = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
cfg.fidelity_cache_root = "/content/fidelity_cache_local"

cfg.p = 2
cfg.seeds = (123,)
cfg.modes = ("DSWD", "HDSWD", "HSW_DSW", "HSW_HDSWD")

cfg.epochs = 100
cfg.eval_every = 10
cfg.sample_every = 5
cfg.run_tag = "bs512_structured_four_modes_v1"

cfg.latent_size = 100
cfg.lr_g = 5e-4
cfg.lr_d = 5e-4
cfg.lr_t = 1e-4

cfg.num_projections = 256
cfg.selector_steps = 1
cfg.use_diversity_reg = True
cfg.diversity_weight = 10.0

cfg.h_num_base = 128
cfg.h_num_bottleneck = 32
cfg.h_base_rank = 512
cfg.h_bottleneck_diversity_weight = 5.0
cfg.h_mix_act = "linear"

cfg.hsw_slicer_dims = (512, 128)
cfg.hsw_slicer_act = "linear"
cfg.hsw_proj_rank = 0

cfg.drive_backup_every = 10
cfg.drive_backup_on_eval = True
cfg.final_full_sync_to_drive = True
cfg.sync_history_on_backup = True
cfg.sync_eval_history_on_backup = True
cfg.sync_samples_on_backup = False
cfg.sync_eval_ckpts_on_backup = False

cfg.resume_if_possible = True
cfg.autoskip_complete = True

print("Experiment root:", experiment_root(cfg))
train_all(cfg)


Experiment root: /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1
[INFO] No valid checkpoint found for DSWD seed=123; starting fresh.
[DSWD][seed=123] epoch 001/100 | G=1.0328 D=1.1806 SW=1.0328 DIV=0.1402 MAC=0.3649 MB_time=0.0394s
[DSWD][seed=123] epoch 002/100 | G=0.6873 D=0.8849 SW=0.6873 DIV=0.0219 MAC=0.1410 MB_time=0.0395s
[DSWD][seed=123] epoch 003/100 | G=1.2803 D=0.9438 SW=1.2803 DIV=0.0181 MAC=0.1246 MB_time=0.0392s
[DSWD][seed=123] epoch 004/100 | G=1.7389 D=0.9190 SW=1.7389 DIV=0.0345 MAC=0.1750 MB_time=0.0392s
[DSWD][seed=123] epoch 005/100 | G=1.4404 D=1.0078 SW=1.4404 DIV=0.0406 MAC=0.1911 MB_time=0.0392s
[DSWD][seed=123] epoch 006/100 | G=1.4901 D=0.7833 SW=1.4901 DIV=0.0350 MAC=0.1771 MB_time=0.0392s
[DSWD][seed=123] epoch 007/100 | G=1.6391 D=0.7264 SW=1.6391 DIV=0.0404 MAC=0.1924 MB_time=0.0393s
[DSWD][seed=123] epoch 008/100 | G=1.7537 D=0.6974 SW=1.7537 DIV=0.0428 MAC=0.1992 MB_time=0.0392s
[DSWD][seed=123] epoch 009/100 | G=1.6228 D=0.7157 SW=1.6

In [24]:
# @title
# Summarize results (best checkpoint and final epoch)
from pathlib import Path
import pandas as pd

base = experiment_root(cfg)
rows_best = []
rows_final = []

for mode_dir in base.iterdir():
    if not mode_dir.is_dir():
        continue
    for seed_dir in mode_dir.iterdir():
        eval_csv = seed_dir / "logs" / "eval_history.csv"
        if not eval_csv.exists():
            continue
        df = pd.read_csv(eval_csv)
        if len(df) == 0:
            continue

        best = df.loc[df["fid"].idxmin()].copy()
        final = df.sort_values("epoch").iloc[-1].copy()

        best["seed"] = seed_dir.name
        final["seed"] = seed_dir.name
        rows_best.append(best)
        rows_final.append(final)

best_df = pd.DataFrame(rows_best).sort_values("fid").reset_index(drop=True)
final_df = pd.DataFrame(rows_final).sort_values("fid").reset_index(drop=True)

print("=== Best over training ===")
display(best_df)

print("\n=== Final epoch ===")
display(final_df)


=== Best over training ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
1,100,seed_42,HDSWD,58.694691,0.041272,0.001359
2,100,seed_123,DSWD,58.962416,0.041094,0.001454
3,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299
4,100,seed_42,DSWD,62.531611,0.043902,0.001393
5,100,seed_123,HSW_DSW,71.982316,0.054797,0.001448
6,100,seed_123,HSW_HDSWD,77.625569,0.057942,0.001832
7,90,seed_123,HDSWD,83.118562,0.044150,0.001078



=== Final epoch ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
1,100,seed_42,HDSWD,58.694691,0.041272,0.001359
2,100,seed_123,DSWD,58.962416,0.041094,0.001454
3,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299
4,100,seed_42,DSWD,62.531611,0.043902,0.001393
5,100,seed_123,HSW_DSW,71.982316,0.054797,0.001448
6,100,seed_123,HSW_HDSWD,77.625569,0.057942,0.001832
7,100,seed_123,HDSWD,249.245444,0.224762,0.002766


In [25]:
# @title
# Run four structured modes with two switches:
# DSWD (none), HDSWD (projector only), HSW_DSW (slicer only), HSW_HDSWD (both)

cfg = TrainConfig()

cfg.output_root = "/content/runs_local"
cfg.drive_output_root = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
cfg.fidelity_cache_root = "/content/fidelity_cache_local"

cfg.p = 2
cfg.seeds = (40,)
cfg.modes = ("DSWD", "HDSWD", "HSW_DSW", "HSW_HDSWD")

cfg.epochs = 100
cfg.eval_every = 10
cfg.sample_every = 5
cfg.run_tag = "bs512_structured_four_modes_v1"

cfg.latent_size = 100
cfg.lr_g = 5e-4
cfg.lr_d = 5e-4
cfg.lr_t = 1e-4

cfg.num_projections = 256
cfg.selector_steps = 1
cfg.use_diversity_reg = True
cfg.diversity_weight = 10.0

cfg.h_num_base = 128
cfg.h_num_bottleneck = 32
cfg.h_base_rank = 512
cfg.h_bottleneck_diversity_weight = 5.0
cfg.h_mix_act = "linear"

cfg.hsw_slicer_dims = (512, 128)
cfg.hsw_slicer_act = "linear"
cfg.hsw_proj_rank = 0

cfg.drive_backup_every = 10
cfg.drive_backup_on_eval = True
cfg.final_full_sync_to_drive = True
cfg.sync_history_on_backup = True
cfg.sync_eval_history_on_backup = True
cfg.sync_samples_on_backup = False
cfg.sync_eval_ckpts_on_backup = False

cfg.resume_if_possible = True
cfg.autoskip_complete = True

print("Experiment root:", experiment_root(cfg))
train_all(cfg)


Experiment root: /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1
[INFO] No valid checkpoint found for DSWD seed=40; starting fresh.
[DSWD][seed=40] epoch 001/100 | G=1.0928 D=5.6909 SW=1.0928 DIV=0.1328 MAC=0.3533 MB_time=0.0394s
[DSWD][seed=40] epoch 002/100 | G=0.8287 D=0.9026 SW=0.8287 DIV=0.0237 MAC=0.1493 MB_time=0.0392s
[DSWD][seed=40] epoch 003/100 | G=1.2265 D=0.9969 SW=1.2265 DIV=0.0199 MAC=0.1333 MB_time=0.0394s
[DSWD][seed=40] epoch 004/100 | G=1.1336 D=1.1002 SW=1.1336 DIV=0.0274 MAC=0.1566 MB_time=0.0396s
[DSWD][seed=40] epoch 005/100 | G=1.1392 D=0.9847 SW=1.1392 DIV=0.0263 MAC=0.1530 MB_time=0.0395s
[DSWD][seed=40] epoch 006/100 | G=1.4028 D=0.8639 SW=1.4028 DIV=0.0320 MAC=0.1702 MB_time=0.0393s
[DSWD][seed=40] epoch 007/100 | G=1.5030 D=0.7470 SW=1.5030 DIV=0.0354 MAC=0.1801 MB_time=0.0395s
[DSWD][seed=40] epoch 008/100 | G=1.4147 D=0.8003 SW=1.4147 DIV=0.0361 MAC=0.1823 MB_time=0.0394s
[DSWD][seed=40] epoch 009/100 | G=1.3394 D=0.7885 SW=1.3394 DIV=0.

In [26]:
# @title
# Run four structured modes with two switches:
# DSWD (none), HDSWD (projector only), HSW_DSW (slicer only), HSW_HDSWD (both)

cfg = TrainConfig()

cfg.output_root = "/content/runs_local"
cfg.drive_output_root = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
cfg.fidelity_cache_root = "/content/fidelity_cache_local"

cfg.p = 2
cfg.seeds = (2026,)
cfg.modes = ("DSWD", "HDSWD", "HSW_DSW", "HSW_HDSWD")

cfg.epochs = 100
cfg.eval_every = 10
cfg.sample_every = 5
cfg.run_tag = "bs512_structured_four_modes_v1"

cfg.latent_size = 100
cfg.lr_g = 5e-4
cfg.lr_d = 5e-4
cfg.lr_t = 1e-4

cfg.num_projections = 256
cfg.selector_steps = 1
cfg.use_diversity_reg = True
cfg.diversity_weight = 10.0

cfg.h_num_base = 128
cfg.h_num_bottleneck = 32
cfg.h_base_rank = 512
cfg.h_bottleneck_diversity_weight = 5.0
cfg.h_mix_act = "linear"

cfg.hsw_slicer_dims = (512, 128)
cfg.hsw_slicer_act = "linear"
cfg.hsw_proj_rank = 0

cfg.drive_backup_every = 10
cfg.drive_backup_on_eval = True
cfg.final_full_sync_to_drive = True
cfg.sync_history_on_backup = True
cfg.sync_eval_history_on_backup = True
cfg.sync_samples_on_backup = False
cfg.sync_eval_ckpts_on_backup = False

cfg.resume_if_possible = True
cfg.autoskip_complete = True

print("Experiment root:", experiment_root(cfg))
train_all(cfg)


Experiment root: /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1
[INFO] No valid checkpoint found for DSWD seed=2026; starting fresh.
[DSWD][seed=2026] epoch 001/100 | G=0.8839 D=1.6723 SW=0.8839 DIV=0.1204 MAC=0.3334 MB_time=0.0393s
[DSWD][seed=2026] epoch 002/100 | G=0.7822 D=0.8189 SW=0.7822 DIV=0.0143 MAC=0.1128 MB_time=0.0392s
[DSWD][seed=2026] epoch 003/100 | G=1.0118 D=1.0988 SW=1.0118 DIV=0.0149 MAC=0.1101 MB_time=0.0395s
[DSWD][seed=2026] epoch 004/100 | G=1.3269 D=1.0134 SW=1.3269 DIV=0.0282 MAC=0.1560 MB_time=0.0392s
[DSWD][seed=2026] epoch 005/100 | G=1.3494 D=0.9831 SW=1.3494 DIV=0.0315 MAC=0.1665 MB_time=0.0392s
[DSWD][seed=2026] epoch 006/100 | G=1.3334 D=0.9006 SW=1.3334 DIV=0.0327 MAC=0.1699 MB_time=0.0392s
[DSWD][seed=2026] epoch 007/100 | G=1.4611 D=0.7999 SW=1.4611 DIV=0.0346 MAC=0.1756 MB_time=0.0394s
[DSWD][seed=2026] epoch 008/100 | G=1.4097 D=0.8150 SW=1.4097 DIV=0.0356 MAC=0.1785 MB_time=0.0393s
[DSWD][seed=2026] epoch 009/100 | G=1.3884 D=0.7

In [27]:
# @title
# Run four structured modes with two switches:
# DSWD (none), HDSWD (projector only), HSW_DSW (slicer only), HSW_HDSWD (both)

cfg = TrainConfig()

cfg.output_root = "/content/runs_local"
cfg.drive_output_root = "/content/drive/MyDrive/cifar10_dsw_runs" if os.path.exists("/content/drive/MyDrive") else "./runs_drive_backup"
cfg.fidelity_cache_root = "/content/fidelity_cache_local"

cfg.p = 2
cfg.seeds = (3,)
cfg.modes = ("DSWD", "HDSWD", "HSW_DSW", "HSW_HDSWD")

cfg.epochs = 100
cfg.eval_every = 10
cfg.sample_every = 5
cfg.run_tag = "bs512_structured_four_modes_v1"

cfg.latent_size = 100
cfg.lr_g = 5e-4
cfg.lr_d = 5e-4
cfg.lr_t = 1e-4

cfg.num_projections = 256
cfg.selector_steps = 1
cfg.use_diversity_reg = True
cfg.diversity_weight = 10.0

cfg.h_num_base = 128
cfg.h_num_bottleneck = 32
cfg.h_base_rank = 512
cfg.h_bottleneck_diversity_weight = 5.0
cfg.h_mix_act = "linear"

cfg.hsw_slicer_dims = (512, 128)
cfg.hsw_slicer_act = "linear"
cfg.hsw_proj_rank = 0

cfg.drive_backup_every = 10
cfg.drive_backup_on_eval = True
cfg.final_full_sync_to_drive = True
cfg.sync_history_on_backup = True
cfg.sync_eval_history_on_backup = True
cfg.sync_samples_on_backup = False
cfg.sync_eval_ckpts_on_backup = False

cfg.resume_if_possible = True
cfg.autoskip_complete = True

print("Experiment root:", experiment_root(cfg))
train_all(cfg)


Experiment root: /content/runs_local/cifar10_dsw_v2/bs512_structured_four_modes_v1
[INFO] No valid checkpoint found for DSWD seed=3; starting fresh.
[DSWD][seed=3] epoch 001/100 | G=1.0014 D=2.5456 SW=1.0014 DIV=0.1374 MAC=0.3608 MB_time=0.0393s
[DSWD][seed=3] epoch 002/100 | G=0.7006 D=0.9333 SW=0.7006 DIV=0.0231 MAC=0.1461 MB_time=0.0393s
[DSWD][seed=3] epoch 003/100 | G=0.8714 D=1.0525 SW=0.8714 DIV=0.0147 MAC=0.1135 MB_time=0.0392s
[DSWD][seed=3] epoch 004/100 | G=1.3840 D=1.0489 SW=1.3840 DIV=0.0270 MAC=0.1549 MB_time=0.0392s
[DSWD][seed=3] epoch 005/100 | G=1.3302 D=0.9907 SW=1.3302 DIV=0.0302 MAC=0.1646 MB_time=0.0392s
[DSWD][seed=3] epoch 006/100 | G=1.5340 D=0.8516 SW=1.5340 DIV=0.0353 MAC=0.1794 MB_time=0.0393s
[DSWD][seed=3] epoch 007/100 | G=1.7381 D=0.7772 SW=1.7381 DIV=0.0410 MAC=0.1944 MB_time=0.0392s
[DSWD][seed=3] epoch 008/100 | G=1.7414 D=0.7031 SW=1.7414 DIV=0.0423 MAC=0.1978 MB_time=0.0393s
[DSWD][seed=3] epoch 009/100 | G=1.7437 D=0.7465 SW=1.7437 DIV=0.0431 MAC=0

In [28]:
# @title
# Summarize results (best checkpoint and final epoch)
from pathlib import Path
import pandas as pd

base = experiment_root(cfg)
rows_best = []
rows_final = []

for mode_dir in base.iterdir():
    if not mode_dir.is_dir():
        continue
    for seed_dir in mode_dir.iterdir():
        eval_csv = seed_dir / "logs" / "eval_history.csv"
        if not eval_csv.exists():
            continue
        df = pd.read_csv(eval_csv)
        if len(df) == 0:
            continue

        best = df.loc[df["fid"].idxmin()].copy()
        final = df.sort_values("epoch").iloc[-1].copy()

        best["seed"] = seed_dir.name
        final["seed"] = seed_dir.name
        rows_best.append(best)
        rows_final.append(final)

best_df = pd.DataFrame(rows_best).sort_values("fid").reset_index(drop=True)
final_df = pd.DataFrame(rows_final).sort_values("fid").reset_index(drop=True)

print("=== Best over training ===")
display(best_df)

print("\n=== Final epoch ===")
display(final_df)

=== Best over training ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_3,DSWD,53.321282,0.036475,0.001425
1,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
2,100,seed_3,HSW_HDSWD,56.851530,0.040016,0.001260
3,100,seed_2026,DSWD,57.273623,0.039733,0.001502
4,100,seed_2026,HSW_HDSWD,58.402174,0.041671,0.001408
5,100,seed_3,HDSWD,58.411206,0.041711,0.001530
6,100,seed_42,HDSWD,58.694691,0.041272,0.001359
7,100,seed_123,DSWD,58.962416,0.041094,0.001454
8,100,seed_40,HSW_DSW,59.035532,0.041350,0.001281
9,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299



=== Final epoch ===


,epoch,seed,mode,fid,kid_mean,kid_std
0,100,seed_3,DSWD,53.321282,0.036475,0.001425
1,100,seed_42,HSW_DSW,54.988746,0.037053,0.001283
2,100,seed_3,HSW_HDSWD,56.851530,0.040016,0.001260
3,100,seed_2026,DSWD,57.273623,0.039733,0.001502
4,100,seed_2026,HSW_HDSWD,58.402174,0.041671,0.001408
5,100,seed_3,HDSWD,58.411206,0.041711,0.001530
6,100,seed_42,HDSWD,58.694691,0.041272,0.001359
7,100,seed_123,DSWD,58.962416,0.041094,0.001454
8,100,seed_40,HSW_DSW,59.035532,0.041350,0.001281
9,100,seed_42,HSW_HDSWD,59.506497,0.042151,0.001299
